In [ ]:
#import scarches as sca
#import squidpy as sq
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import os
import time
import pickle
from itertools import chain
import itertools
from tqdm.auto import tqdm
import anndata as ad
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
from adjustText import adjust_text

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

# FUNCTIONS

In [ ]:

###=======================
def create_celltype_cluster_colormap(adata,color_var):
    
    ## CREATE CLUSTER COLORING
    clusters=np.sort(adata.obs[color_var].unique()) #cellt_freq_df.reset_index()[color_var] 
    
    # Extract cell types from cluster names
    cell_types = np.unique([c.split('_')[0] for c in clusters])
    
    # Assign a base color to each cell type
    base_colors = sns.color_palette('tab10', len(cell_types))
    cell_type_color_mapping = dict(zip(cell_types, base_colors))
    
    lightness_threshold=1
    
    # Create the final color mapping
    clusters_color_mapping = {}
    for cell_type in cell_types:
        
        # Get all clusters for the current cell type
        cell_type_clusters = [c for c in clusters if c.startswith(cell_type)]
    
        if len(cell_type_clusters)>1:
    
            # Create a blended palette for the current cell type
            shades = sns.blend_palette([cell_type_color_mapping[cell_type], (1, 1, 1)], 
                                       n_colors=(len(cell_type_clusters)+1), as_cmap=False, input="rgb")
            '''
            # Adjust the lightness of the lightest color
            shades = [(max(min(color[0], lightness_threshold), 0), 
                       max(min(color[1], lightness_threshold), 0), 
                       max(min(color[2], lightness_threshold), 0)) for color in shades]
            '''
            
            # Assign shades to clusters
            for i, cluster in enumerate(cell_type_clusters):
                clusters_color_mapping[cluster] = shades[i]
        
        if len(cell_type_clusters)==1:
            clusters_color_mapping[cell_type] = cell_type_color_mapping[cell_type]
    return clusters_color_mapping


##### =============================================
## Based on the annotated low-level celltypes, add a new column to adata.obs with the final high-level celltypes
#  (as one Macrophage cluster was given Neutrophil in the high-level clustering in xen_4_manual_annotation.ipynb)
def add_celltype_as_prefix(adata_,annotation_coln):

    bdata=adata_.copy()
    
    celltype_dict={'Macrophage':['Mac_4',
                                'Mac_5',
                                'Mac_3',

                                 'Mac_C1Q',
                                 'Mac_HMOX1',
                                 'Mac_IL10',
                                 'Mac_TREM2',

                                 'C1Q', 
                                 'HMOX1',
                                 'IFN_resp.', 
                                 'IL10+/TNFAIP3+', 
                                
                                 'PLIN2hi/TREM1hi',
                                 'S100A8/IL1B+', 
                                 'S100A8/IL1B-', 
                                 'TREM2hi'],
                   
                   'T-cell':[
                               'CD4_Exh.', 
                              'CD4_TCM', 
                              'CD4_TEM',
                              'CD4_Tregs', 
                              'CD8_IFN_resp.',
                              'CD8_TEFF', 
                              'CD8_TEM', 
                              'MAIT',
                              'Proliferating',
                               'Unknown',
                                'Teff-cell'],
                   
                   
                   'B2-cell':['B2-cell'],

                   'Neutrophil':['Neutrophil'],
                   
                   'Endothelial':['EC_1',
                                  'EC_2'],

                   'Plasma-cell':['Plasma-cell'],

                   'Mast-cell':['Mast-cell'],
                
                   
                   'VSMC':['VSMC-contr', 
                            'VSMC-synth', 
                            'VSMC-foam', 
                            'VSMC-macrophage-like', 
                            'VSMC-adipocyte-like', 
                            'VSMC-osteochondrogenic', 
                            'VSMC-myofb-like',
                            'VSMC_3',
                            'VSMC_2',
                            'VSMC-ec-like',
                            'VSMC-msc-like', 
                            'Modulated_VSMCs']}

    high_level_celt_map={low_level_cellt: high_level_cellt for high_level_cellt in celltype_dict.keys() for low_level_cellt in celltype_dict[high_level_cellt]}


    
    #adata.obs['final_high_level_celltype']=adata.obs['wilcoxon_high_level_celltype'].values
    bdata.obs['final_high_level_celltype']=bdata.obs[annotation_coln].map(high_level_celt_map)

    bdata.obs[annotation_coln]=bdata.obs[annotation_coln].astype(str).replace({'Modulated_VSMCs':'VSMC_modulated',
                                                                              'Mac_':'Macrophage_',
                                                                              'EC_':'Endothelial_',
                                                                              'VSMC-':'VSMC_'},regex=True)

    
    bdata.obs[f'{annotation_coln}_final']=bdata.obs[f'{annotation_coln}'].astype(str).values


    filt=(bdata.obs[annotation_coln].str.split('_',expand=True)[0]) != (bdata.obs['final_high_level_celltype'])
    cols_to_join=['final_high_level_celltype',annotation_coln]
    bdata.obs.loc[filt,f'{annotation_coln}_final']=bdata.obs.loc[filt,cols_to_join].astype(str).T.agg('_'.join)
    
    #adata.obs.loc[filt,'final_high_level_celltype'] + '_' +\
    #                                                adata.obs.loc[filt,annotation_coln].str.split('_',expand=True)[1]
    

    ## REPLACE FOR FINAL CLUSTER NAMES
    bdata.obs[f'{annotation_coln}_final']=bdata.obs[f'{annotation_coln}_final'].astype(str).replace({'VSMC_3':'VSMC_inflamed',
                                                                              'VSMC_myofb-like':'VSMC_fibroblast-like',
                                                                              },regex=True)

    return bdata

'''
adata=adata_dict['Panel1']
annotation_coln='manual_res_deseq2_annot_low_level_celltype'

a=add_celltype_as_prefix(adata,annotation_coln)
print(a.obs['manual_res_deseq2_annot_low_level_celltype_final'].unique())
print(a.obs['final_high_level_celltype'].unique())
'''

##### =============================================
## Return the set of parameters in search_space as a string, to distinguish models trained with different parameter sets
def return_parameter_set(final_search_space):
    
    # Create a list of dictionaries where each dictionary represents a set of hyperparameters
    parameter_combinations = list(itertools.product(*final_search_space.values()))
    hyperparameter_sets = [{key: value for key, value in zip(final_search_space.keys(), combination)} for combination in parameter_combinations]
    
    # Print the list of hyperparameter sets
    for n,hyperparameters in enumerate(hyperparameter_sets):
        ## Create string of the parameters
        pairs=[f"{key}:{value}" for key, value in hyperparameters.items()]
        param_set_string='-'.join(pairs)

    return param_set_string





# LOAD BATCH CORRECTED DATA & ANNOTATED DATA

In [ ]:
from scipy.io import mmread

proc_dir='../../../xenium_data/processed_data/baysor_processed_output'
scale_param='10'
avg_assignment_conf_thr=0.75

adata_dict={}

for panel in ['Panel1','Panel2'][:]:

    

    ## Save batch corrected data
    #fn=os.path.join(proc_dir,f'filtered_batch_corr_{panel}_cells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}.h5ad')
    fn=os.path.join(proc_dir,f'filtered_batch_corr_annotated_{panel}_cells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}.h5ad')
    fn=os.path.join(proc_dir,f'filtered_batch_corr_annotated_{panel}_cells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}_.h5ad')
    #fn=os.path.join(proc_dir,f'filtered_batch_corr_annotated_gnn_embeddings_{panel}_cells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}.h5ad')
    #fn=os.path.join(proc_dir,f'filtered_batch_corr_annotated_knn_neigh_{panel}_cells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}.h5ad')
    adata=sc.read_h5ad(fn)

    adata.obsm["spatial"] = adata.obs[["x", "y"]].copy().to_numpy()
    #adata.uns["spatial"] = adata.obs[["x", "y"]].copy().to_numpy()

    ## Load Sample regions to .obs
    fn=os.path.join(proc_dir,f'filtered_{panel}_cells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}.h5ad')
    adata_=sc.read_h5ad(fn)
    adata.obs['sample_region']=adata_.obs.loc[adata.obs.index,'sample_region'].values
    adata.layers['norm_by_area']=adata_[adata.obs.index,:].layers['norm_by_area']


    ## Load and add region distances to .obs if it has been calculated
    try:
        fn=os.path.join(proc_dir,'region_distances',f'{panel}_region_distances_norm.csv')
        region_distances_with_norm=pd.read_csv(fn,index_col=0)
        adata.obs=pd.concat([adata.obs,region_distances_with_norm.loc[adata.obs.index,:]],axis=1)
    except FileNotFoundError:
        pass


    #adata.layers=adata_.layers
    del adata_

    ## Load previously acosh normalised counts 
    #fn=os.path.join(proc_dir,f'filtered_batch_corr_annotated_{panel}_cells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}_acosh.mtx')
    #acosh_matrix=mmread(fn)
    #adata.layers['acosh']=acosh_matrix.T.tocsr()
    
    print(adata.layers)
    adata_dict[panel]=adata



### DEFINE SCVI REPRESENTATION PARAMETERS
if scale_param=='5':
    final_search_space={
            "n_hidden":[256],
            "n_latent":[20], #20
            "n_layers": [1], #2
            'gene_likelihood':['zinb'],
            'max_epochs':[600]}
    
if scale_param=='10':
    final_search_space={
            "n_hidden":[256],
            "n_latent":[20], #20
            "n_layers": [1], #2
            'gene_likelihood':['zinb'],
            'max_epochs':[600]}
    
if scale_param=='15':

    final_search_space={
            "n_hidden":[128],
            "n_latent":[20], #20
            "n_layers": [1], #2
            'gene_likelihood':['zinb'],
            'max_epochs':[600]}

if scale_param=='5|10|15':
        final_search_space={
            "n_hidden":[256],
            "n_latent":[20], #20
            "n_layers": [1], #2
            'gene_likelihood':['zinb'],
            'max_epochs':[600]}

param_set_string = return_parameter_set(final_search_space)

batch_corr_method='scVI'

neighbors_key="X_scVI_"+param_set_string 

# ADD PATIENT METADATA TO ANNDATA

In [ ]:
data_dir="../../xenium_data"

fn=os.path.join(data_dir)
metadata=pd.read_excel(f'{data_dir}/Xenium_Patientendaten_MVB_NS22052024_toextern.xlsx',header=1)
metadata.columns=[x.strip() for x in metadata.columns]
sample_pat_name_df=pd.read_excel(f'{data_dir}/sample_name_patient_name_dictionary.xlsx')

# Extract stability information for patients P5-P12
sample_pat_name_df['pat_id']=sample_pat_name_df['patient_id'].str.split('_',expand=True)[0]
sample_pat_name_df=sample_pat_name_df.set_index('pat_id',drop=True)
sample_pat_name_df=sample_pat_name_df[~sample_pat_name_df.index.duplicated(keep='first')]

## CREATE column with mathcing pat id values as in the anndata + shorten some column names
metadata['condition']=metadata['sample_id'].str.split('_',expand=True)[1]
metadata['condition']=metadata['condition'].replace({'H':'Control','D':'Plaque'},regex=True)
metadata['pat_id']=metadata['sample_id'].str.split('_',expand=True)[0]
metadata['symptomatic']=metadata['Symptomatic']
metadata['Smoking']=metadata['Smoking'].replace({'Ex':'ex'})
metadata['Smoking_ever']=metadata['Smoking'].replace({'ex':'yes'})
metadata=metadata.set_index('pat_id',drop=True)

#metadata['stability']='nan'
#metadata.loc[sample_pat_name_df.index,'stability']=sample_pat_name_df['stability'].values



for panel in ['Panel1','Panel2'][:]:
    print(panel)
    adata=adata_dict[panel]

    #metadata=metadata.set_index('pat_id',drop=False)
    for meta_coln in ['symptomatic','Age','Sex','Hypertension', 'Diabetes', 'Smoking', 'Smoking_ever','Dyslipidemia',
                       'CHD', 'Atrial fibrilation', 'PAD', 'COPD', 'Dialysis', 'Size (cm)',
                       'Weight (kg)', 'BMI', 'SAPT', 'Statin']:
        
        mat_coln_dict=metadata[meta_coln].to_dict()
        adata.obs[meta_coln]=adata.obs['patient'].map(mat_coln_dict)

    ## Set SAMPLE-ID as index to add stability of the samples
    metadata_=metadata.copy()
    metadata_['sample_id']=panel+'_' + metadata['sample_id'].replace({'_H':'_Ctrl','_D':'_Pl'},regex=True)
    metadata_=metadata_.set_index('sample_id',drop=False)
    
    for meta_coln in ['stability']:
        
        mat_coln_dict=metadata_[meta_coln].to_dict()
        adata.obs[meta_coln]=adata.obs['original_sample'].astype(str).map(mat_coln_dict)
 
    adata.obs['stability']=adata.obs['stability'].replace({'+':'stable','-':'unstable'})
    #adata.obs['condition']=adata.obs['condition'].replace({'H':'Healthy','D':'Diseased'},regex=True)
    adata.obs['symptomatic']=adata.obs['symptomatic'].replace({'a':'asympt.','s':'sympt.'},regex=True)
    
    adata_dict[panel]=adata
    print(adata.obs['stability'].unique())

# OVERLAPPING GENES WITH BULK SMOKING DEGS

## FIG.6/FIG.4_Suppl.FIG.K - Bulk Volcano + concordant - discordant Xenium genes highlighted

In [ ]:
## LOAD BULK DGE RESULTS
diff_smoke=pd.read_excel('../../../data//DIFF-EX_Smoking_Yes_Ex_vs_No_Diseased_only.xlsx',index_col=0)


from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
from matplotlib.font_manager import FontProperties
import matplotlib.patheffects as pe

log2FC_thr=0.5
fdr_lev=0.05
padj_thr=-np.log10(fdr_lev)
panels=['Panel1','Panel2']

de_methods=['deseq2','wilcoxon']
cellt_var='final_low_level_celltype'
coln_for_dge_list=['Smoking_ever']



for coln_for_dge in coln_for_dge_list[:]:
    
    for de_method in de_methods[:1]:

        cellt_df_list=[]

        diff_smoke['DGE_Xenium']='no'
        diff_smoke['DGE_Xenium_FC_dir']='nan'
  
     
        celltype_df=diff_smoke.loc[(diff_smoke['padj']<fdr_lev)\
                                  &(diff_smoke['log2FoldChange'].abs()>log2FC_thr)].copy()

        
        for panel in ['Panel1','Panel2'][:]:
    
            print(panel)
            adata=adata_dict[panel]
            #adata=adata[~adata.obs['original_sample'].str.contains('_H'),:].copy()
            
            ##====== RUN DGE AND TRY TO ANNOTATE CLUSTERS BASED ON OVERLAPPING MARKER GENES AND DE GENES ======####
         
            print(coln_for_dge)
    
            if coln_for_dge in ['Smoking_ever','Hypertension', 'Diabetes', 'Dyslipidemia','CHD','PAD','Sex','sample_region','stability']:
                adata_=adata[(~adata.obs['original_sample'].str.contains('_Ctrl'))\
                              &(~adata.obs['sample_region'].str.contains('Thrombus')),:].copy()
                #adata_=adata.copy()#[~adata.obs['original_sample'].str.contains('_H'),:].copy()
         
            
                fn=os.path.join(proc_dir,'xenium_DGE','all_cells',f'{panel}_cells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}_{de_method}_DGE_across_celltypes_within_{coln_for_dge}.csv')
                dge_df_concat=pd.read_csv(fn,index_col=0)
    
                dge_df_concat=dge_df_concat[~dge_df_concat['cluster'].str.contains('Lumen')]

                
                dge_df_concat_ = dge_df_concat[#(dge_df_concat['gene'].isin(diff_smoke.index))\
                                          (dge_df_concat['padj']<fdr_lev)\
                                          &(dge_df_concat['log2FC'].abs()>log2FC_thr)]
                       

                ## Subset BULK results to significant genes that are overlapping with Xenium genes
                diff_smoke_=diff_smoke[(diff_smoke['padj']<0.05)\
                                        &(diff_smoke['log2FoldChange'].abs()>log2FC_thr)
                                        &(diff_smoke.index.isin(dge_df_concat_['gene']))]
        
                ## Rename padj to padj_bulk
                diff_smoke_.columns=[c if c!='padj' else 'padj_bulk' for c in diff_smoke.columns ]

                celltype_df.loc[celltype_df['Symbol'].isin(dge_df_concat['gene_hgnc'].unique()),'DGE_Xenium']='yes'        
                celltype_df.loc[celltype_df['Symbol'].isin(dge_df_concat['gene_hgnc'].unique()),'DGE_Xenium_FC_dir']='discordant'

              
    
                ## LOOP OVER EACH CELLTYPE IN XENIUM DGE DATAFRAME, AND EXRACT THE LOG2FC DIRECTIONS FOR EACH GENE 
                ## + ADD COLUMN INDICATING THE RELATION OF THE LOG2FC'S OF THE TWO METHODS
                for cellt,cellt_df in dge_df_concat_.groupby('celltype'):

                    clusts=cellt_df['cluster'].astype(str).unique().tolist()
                    clusts.sort()
    
                    #print(celltype)
                    #print(celltype_cells[coln_for_dge].value_counts())
                    #print(pd.crosstab(celltype_cells[coln_for_dge],celltype_cells['original_sample']))
                    #print()
                    for cluster in (clusts):
    
                    
                        cellt_df = cellt_df.set_index('gene').drop(columns=['pvalue']).sort_values(by='padj',ascending=True)
                        cellt_df = pd.concat([cellt_df,diff_smoke_.loc[list(set(cellt_df.index)&set(celltype_df.index)),\
                                                                        ['padj_bulk','log2FoldChange','Symbol']]],axis=1)
                    
                        cellt_df['log2FC']=cellt_df['log2FC']*-1
                        #cellt_df = cellt_df.loc[np.sign(cellt_df['log2FC'])==np.sign(cellt_df['log2FoldChange']).values,:]
                    
                        cellt_df['sign_-log10_FDR_Xenium']=np.sign(cellt_df['log2FC']) * -np.log10(cellt_df['padj'])
                        cellt_df['sign_-log10_FDR_Bulk']=np.sign(cellt_df['log2FoldChange']) * -np.log10(cellt_df['padj_bulk'])
        
        
        
        
                        # Define the scenarios
                        cellt_df['scenario'] = np.select([
                                                            (cellt_df['sign_-log10_FDR_Xenium'] > 0) & (cellt_df['sign_-log10_FDR_Bulk'] > 0),
                                                            (cellt_df['sign_-log10_FDR_Xenium'] < 0) & (cellt_df['sign_-log10_FDR_Bulk'] < 0),
                                                            (cellt_df['sign_-log10_FDR_Xenium'] > 0) & (cellt_df['sign_-log10_FDR_Bulk'] < 0),
                                                            (cellt_df['sign_-log10_FDR_Xenium'] < 0) & (cellt_df['sign_-log10_FDR_Bulk'] > 0),
                                                        ],
                                                        ['Both Up', 
                                                         'Both Down', 
                                                         'Up-Xenium, Down-Bulk', 
                                                         'Down-Xenium, Up-Bulk'],
                                                        
                                                        default='Other'
                                                    )
                        cellt_df['panel']=panel
        
                        cellt_df_list.append(cellt_df)

        

        ## Concatenate celltype DGE dataframes & extract if the log2FC are concordant or discordant between Xenium and Bulk
        xen_cellt_concat=pd.concat(cellt_df_list,axis=0)       
        a=(xen_cellt_concat[xen_cellt_concat['scenario'].str.contains('Both')])
        b=(xen_cellt_concat[xen_cellt_concat['scenario'].str.contains('Xenium')])
        #a[a['gene_hgnc'].duplicated(keep=False)].sort_values('gene_hgnc')
        #a=a.drop_duplicates(subset=['panel','Symbol'])

        celltype_df.loc[list(set(a.index)),'DGE_Xenium_FC_dir']='concordant'
        celltype_df.loc[list(set(b.index)),'DGE_Xenium_FC_dir']='discordant'
        
        
        celltype_df['-log10_FDR']=-np.log10(celltype_df['padj'])
        #celltype_df['log10FC'] = celltype_df['log2FoldChange'] * np.log10(2)

        x='log2FoldChange'
        #x='log10FC'
        y='-log10_FDR'
        
        x_axis_limit=max(abs(celltype_df[x].min()),abs(celltype_df[x].max()))*1.05
        y_axis_limit=celltype_df[y].max()*1.05
          

        #print(f"{celltype} cells: {celltype_cluster_cells.shape[0]} - Num of cells in clust {cluster}: - {celltype_cells.shape[0]} - ")

        for direction in ['concordant','discordant']:
            print(direction)
        
            #ax=fig.add_subplot(nrows,ncols,n+1)
            fig,ax=plt.subplots(1,1,figsize=(6,5))
            
            ## SHOW ONLY GENES THAT ARE CONCORDANT ACROSS BOTH BULK AND XENIUM REGARDING LOG2FC
            gene_names_to_show = celltype_df.loc[celltype_df['DGE_Xenium_FC_dir']==direction,'Symbol'].tolist()
            #gene_names_to_show = celltype_df.loc[celltype_df['DGE_Xenium_FC_dir']!='nan','Symbol'].tolist()
    
            colors = celltype_df[x].apply(lambda x: 'red' if x> 0 else 'blue')
            alphas= [1.0 if celltype_df['Symbol'].iloc[i] in gene_names_to_show else 0.05 for i in range(len(celltype_df)) ]
            
            ax.scatter(celltype_df[x],celltype_df[y],s=5,edgecolors='black',alpha=alphas,linewidths=0.5,c=colors)
            '''
            sns.scatterplot(data=celltype_df,
                            x=x,
                            y=y,
                            s=20,
                            edgecolors='black',
                            alpha=alphas,
                            linewidths=0.5,
                            #c=colors,
                            hue='DGE_Xenium_FC_dir',
                            ax=ax,
                           )
            '''
            #ax.axvline(log2FC_thr,)
            ax.vlines(x=[-log2FC_thr,log2FC_thr], ymin=0,ymax=y_axis_limit,color='k', ls='--',alpha=0.3)
            ax.axhline(padj_thr,color='k', ls='--',alpha=0.3)
            
            #ax.set_xlim(-x_axis_limit,x_axis_limit)
            x_axis_limit=5
            ax.set_xlim(ax.get_xlim()[0],x_axis_limit)
            ax.set_ylim(padj_thr*0.6,y_axis_limit)
    
            #subplot_title=f"{coln_for_dge.replace('_',' ').capitalize()}: {cluster.replace('_',' ').replace('vs','vs.')}"
            subplot_title = f"{coln_for_dge.replace('_',' ').capitalize()}: {cluster.split('_')[-1]} vs. {cluster.split('_')[0]}"
            ax.set_title(subplot_title,fontweight='bold',fontsize=14)
            ax.set_xlabel(r'$\mathbf{log_2 \, Fold \, change}$' ,fontsize=14)#fontweight='bold')
            #ax.set_xscale('log')
            ax.set_ylabel(r'$\mathbf{-log_{10} \, FDR}$',fontsize=14)#fontweight='bold')
            
    
            texts = [
                ax.text(
                    x=celltype_df[x].iloc[i],
                    y=celltype_df[y].iloc[i],
                    horizontalalignment='center',
                    verticalalignment='bottom',
                    fontsize=7,
                    s=celltype_df['Symbol'].iloc[i],
                    color='red' if celltype_df[x].iloc[i] > 0 else 'blue',  # Set label color
                    path_effects=[pe.withStroke(linewidth=0.5, foreground='black')],
                    bbox=dict(
                        boxstyle="round,pad=0.1",
                        edgecolor="black",
                        facecolor="white",
                        alpha=0.2
                    )  # Add bounding box
                )
    
                for i in range(len(celltype_df))
                if celltype_df['Symbol'].iloc[i] in gene_names_to_show
            ]
            
            adjust_text(texts,
                    force_text=(0.5,0.5),
                    arrowprops=dict(arrowstyle="-", color='k', lw=1,alpha=0.5))
    
            ## Add text of FDR line
            '''
            ax.text(ax.get_xlim()[0]*0.65,
                    #-x_axis_limit*0.75,
                    padj_thr*0.8,f'FDR: {fdr_lev} ',fontsize=12,fontweight='bold',
                    rotation=0,horizontalalignment='center', 
                    verticalalignment='center',rotation_mode='default')
            '''            
            #fig.suptitle(f"Overlapping Xenium {' & '.join(panels)} DGE genes \nwith Bulk DGE genes - {de_method}",fontweight='bold',fontsize=13,y=1.01)
            #plt.grid(alpha=0.2)
            plt.tight_layout()

            if direction=='concordant':
                dir_path=os.path.join(proc_dir,'figure_plots','Fig6','DGE_comparison_with_bulk')
            
            if direction=='discordant':
                #dir_path=os.path.join(proc_dir,'figure_plots','SFigS')
                dir_path=os.path.join(proc_dir,'figure_plots','Fig4','DGE_comparison_with_bulk')
            
            os.makedirs(dir_path,exist_ok=True)
            #fn=os.path.join(proc_dir,'xenium_plots','DGE_volcano_plots',f'{panel}_DGE_{de_method}_contrast_by_{coln_for_dge}.png')
            fn=os.path.join(dir_path,f'cells_scale_{scale_param}_asg_conf_{avg_assignment_conf_thr}_{de_method}_{coln_for_dge}_bulk_volcano_plot_overlapping_{direction}_Xenium_genes.png')
            plt.savefig(fn, bbox_inches='tight',dpi=300)
            

## FIG.6 - ATM & TGFB1R - Pat. 7&10

In [ ]:
## LOAD BULK DGE RESULTS
diff_smoke=pd.read_excel('../../../data//DIFF-EX_Smoking_Yes_Ex_vs_No_Diseased_only.xlsx',index_col=0)

import squidpy as sq
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
from matplotlib.font_manager import FontProperties
import matplotlib.patheffects as pe

log2FC_thr=0.5
fdr_lev=0.05
padj_thr=-np.log10(fdr_lev)
panels=['Panel1','Panel2']

de_methods=['deseq2','wilcoxon']
cellt_var='final_low_level_celltype'
coln_for_dge_list=['Smoking_ever']





gene_ids=['ATM','TGFBR1']
orig_samples=['Panel1_P7_Pl','Panel1_P10_Pl']

count_layers_dict={'raw_counts':'Raw counts',
                   'norm_by_area':'Normalised expression'}


for panel in ['Panel1','Panel2'][:]:
    
    print(panel)

    adata=adata_dict[panel]

    #top_autocorr=(adata.uns["moranI"]["I"].sort_values(ascending=False).head(num_view).index.tolist())
    #bot_autocorr=(adata.uns["moranI"]["I"].sort_values(ascending=True).head(num_view).index.tolist())

    for count_layer in [*count_layers_dict][1:]:
        
        for gene_id_hgnc in gene_ids[:]:
    
            if gene_id_hgnc in adata.var['HGNC'].unique():
                
                gene_id_ens=adata.var[adata.var['HGNC']==gene_id_hgnc].index.tolist()[0]
                
                print(gene_id_hgnc,gene_id_ens)
                
                ## Extract smoker and never smoker plaque sample ids
                smoking_df=adata.obs.drop_duplicates(subset=['original_sample'])[['original_sample','Smoking_ever']].set_index('original_sample')#.to_dict()#.value_counts().res

                smoking_df = smoking_df.loc[orig_samples,:]
                                        #smoking_df.index.str.contains('Pl')]
                
                for smoke_status,samples_ in smoking_df.groupby(['Smoking_ever']):
           

                    ncols=2
                    nrows=int(np.ceil(len(orig_samples)/ncols))
                    fig=plt.figure(figsize=(6*ncols,4*nrows))
                
                    for n,orig_sample in enumerate(samples_.index.tolist()):
                        ax=fig.add_subplot(nrows,ncols,n+1)
                        
                        adata_=adata[adata.obs['original_sample']==orig_sample,:].copy()
                        '''
                        sq.pl.spatial_scatter(adata_, shape=None,
                                              library_key='original_sample',
                                              color=gene_id_ens, 
                                              size=1, cmap="Reds", 
                                              #layer='raw_counts',
                                              layer=count_layer,
                                              #axis_label='original_sample',
                                              ax=ax,

                                              #vmax='p99',
                                              wspace=0.001,
                                              figsize=(5, 4),
                                              #save=f"{proc_dir}/",
                                             )
                        '''
                        expr=adata_.layers[count_layer].A[:,(adata_.var['HGNC']==gene_id_hgnc).values].flatten()
                        
                        scatter= ax.scatter(adata_.obs['x'],
                                            adata_.obs['y'],
                                            #vmin=np.quantile(adata_.layers[count_layer].A,0.01),
                                            #vmax=np.quantile(adata_.layers[count_layer].A,0.99),
                                            vmax=np.quantile(expr,0.99),
                                            c=expr,
                                            cmap='Reds',
                                            s=1,
                                            #edgecolor='k',
                                           )

                        # Create scatter plot with color mapping
                        ax.invert_yaxis()
                        
                        # Add colorbar
                        cbar = plt.colorbar(scatter, ax=ax)
                        cbar.set_label(count_layers_dict[count_layer]) 

                        sample_name=f"{orig_sample.split('_')[1].replace('P','Patient_')} {orig_sample.split('_')[2].replace('Pl','Plaque')}"
                        ax.set_title(f"{sample_name} - {adata_.var.loc[gene_id_ens,'HGNC']}")
                        ax.spines[['top','bottom','left','right']].set_visible(False)
                        ax.set_xlabel(None)
                        ax.set_ylabel(None)
                        ax.axis('off')
    
                    #plt.tight_layout(w_pad=0.01)
                    plt.subplots_adjust(wspace=0.3)

                    ## Save plot                          
                    dir_path=os.path.join(proc_dir,'figure_plots','Fig6','smoking_genes_spatial_scatterplot_final',f'smoker_ever_{smoke_status[0]}')
                    os.makedirs(dir_path,exist_ok=True)
                    
                    fn=os.path.join(dir_path,f"{panel}_{adata_.var.loc[gene_id_ens,'HGNC']}_{count_layer}.png")
                    fig.savefig(fn, bbox_inches='tight',dpi=300)
                    #plt.close()